In [1]:
import ROOT as rt
import uproot
import numpy as np
from collections import OrderedDict
import os
import sys
from array import array
sys.path.append(os.getcwd().replace('scripts', 'lib'))

import tdrstyle
a = tdrstyle.setTDRStyle()

print(sys.version)

Welcome to JupyROOT 6.30/02
3.9.12 (main, Jul 11 2023, 14:37:47) 
[GCC 11.3.0]


# load signal ntuples

Load signal ntuples with 4 different LLP lifetimes (0.1, 1, 10, 100m) that contain generator-level LLP information

In [2]:
fpath =OrderedDict()
tree = OrderedDict()

mass = [40]

OLD_CTAU = np.array([100, 1000, 10000, 100000])#in mm

path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/'

for ct in OLD_CTAU:
    key = 'MC_40_'+str(ct)                       
    fpath[key] = path +  'ggH_HToSSTobbbb_MH-125_MS-40_ctau-'+str(ct)+'_137000pb_weighted.root'

    
NEvents = {}
NEvents_genweight = {}
for k,v in fpath.items():
    root_dir = uproot.open(v) 
    if not root_dir: 
        print(k, "zombie")
        continue
    tree[k] = root_dir['MuonSystem']


# Ex0 - calculate the LLP proper decay length from the LLP four momentum

Here you will calculate the LLP proper decay length by converting from the lab frame decay length to the decay length in the LLP's rest frame, with the following equation:

$d = \beta\gamma\tau$, 

where d is the lab frame decay length, $\beta$ is the LLP velocity, $\gamma$ is the LLP Lorentz factor, and $\tau$ is the LLP proper decay lifetime.

You will use the following branches in the ROOT tree 
* `gLLP_decay_vertex_x`, in units of cm
* `gLLP_decay_vertex_y`, in units of cm
* `gLLP_decay_vertex_z`, in units of cm
* `gLLP_beta` , ranges from 0 to 1

and you can access a branch by doing `T['branch name'].array()` below.

For example to access `gLLP_beta` with `T['gLLP_beta'].array()`. 

The shape of the LLP-level branches is (number of LLPs, 2), because there are two LLPs per event.

Fill in the righthand side of each equation below with the 4 branches mentioned to calculate the proper decay length

In [ ]:
gLLP_ctau = {}
for k, T in tree.items():
    gLLP_decay_vertex = np.sqrt(T['gLLP_decay_vertex_x'].array()**2 + T['gLLP_decay_vertex_y'].array()**2 + T['gLLP_decay_vertex_z'].array()**2)
    gLLP_gamma = 1.0/np.sqrt(1.0 - T['gLLP_beta'].array()**2)
    gLLP_ctau[k] = gLLP_decay_vertex/(T['gLLP_beta'].array()*gLLP_gamma)

**Solution for Ex0 (proper decay length).** The lab-frame decay length of each LLP is the distance from the primary vertex, which is the origin, to its decay vertex: `sqrt(x^2 + y^2 + z^2)`, in cm. The proper decay length follows from `d = beta * gamma * ctau`, so `ctau = d / (beta * gamma)`, where `gamma = 1/sqrt(1 - beta^2)`. The branches are in cm, so the result is in cm.

Physics intuition: a moving LLP covers more lab distance than it would at rest. Dividing by `beta * gamma` removes that time dilation and leaves `c * tau`, the decay length in the LLP's own rest frame.

Common pitfalls: (1) build `gamma` from `gLLP_beta`, which is the velocity in units of c; (2) the LLP branches have shape (events, 2), so keep the operations element-wise and do not sum over the LLP axis here; (3) the decay length assumes the LLP is produced at the origin, which holds for these generator-level samples; (4) the branches are in cm while the file names are in mm, so 1000 mm in a file name means 100 cm in the branches.

# Ex1 - plot the LLP proper decay length

In this exercise, you will plot the `gLLP_ctau` array for 1m and 10m sample.

You will then perform an exponential fit to verify that the shape is exponential and linear in logY scale.

From the exponential fit, you can extract the mean proper decay length of the LLP and verify that it agrees with what you expect from the sample (1m or 10m)

You have to replace the `REPLACE_ME` variable below to fill in the correct variable to extract the fitted ctau by using the variable `fitted_slope`. 

Note that the unit in the tree branches are in cm, while the unit in the ROOT file names are in mm.

In [ ]:
%%time

Nevents_todraw = 10000
c = rt.TCanvas('c','c', 800, 600)
h = {}

for i, k in enumerate([ 'MC_40_1000', 'MC_40_10000']):
    h[k] = rt.TH1D('', '', 100, 0, 10000)
    h[k].SetXTitle('LLP c#tau [cm]')
    h[k].SetYTitle('fraction of events')
    for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0])
    h[k].Scale(1./h[k].Integral())  #normalize histogram to 1
    h[k].SetLineColor(i+1)
    h[k].Draw('hist same')

    ####### perform exponential fit ######
    h[k].Fit('expo')
    fit = h[k].GetFunction('expo')
    fitted_slope = fit.GetParameter(1) # expo is exp(p0 + p1*x), so p1 = -1/ctau, in cm
    print("sample: {}, fitted ctau [mm]: {}".format(k, -10.0/fitted_slope))
    #######################################
    
c.SetLogy()
c.Draw()

**Solution for Ex1 (fit the decay length).** The histogram of `gLLP_ctau` should fall off exponentially, which is a straight line on the log scale. ROOT's `expo` model is `exp(p0 + p1 * x)`, so the slope `p1` equals `-1/ctau`. The code reads `p1` with `fit.GetParameter(1)`, inverts it, and multiplies by -10 to convert from cm to mm, which matches the label and the file names.

Common pitfalls: (1) the slope is negative, so the minus sign is needed to get a positive lifetime; (2) the histogram is filled in cm, so the mm answer has a factor of 10, not 1/10; (3) the histogram bins are 100 cm wide, which is coarse for the 1 m sample (its whole distribution sits in only a few bins), so expect only a rough agreement for that sample.

# Ex2 - reweight LLP lifetime 

In this exercise you will learn how to reweight the LLP lifetime.

In these signal events, there are two LLPs and the decay position of the two LLPs in each event are independent, and each LLP decays with an exponential probability, the distribution of events is simply the product of the two LLP decay probabilities:
\begin{equation}
  p(t_1, t_2 | \tau) = \frac{1}{\tau^2}\exp^{-t_1/\tau}\exp^{-t_2/\tau}
\end{equation}
where $\tau$ is the mean proper decay length of the LLPs, and $t_1$ and $t_2$ are the lifetimes of the first LLP and second LLP in their own rest frame respectively, which are the `gLLP_ctau` variable that you have calculated in Ex1.

To obtain a sample with lifetime $\tau_{new}$ from a sample with lifetime $\tau_{old}$, we
assign a weight, which is the ratio between equation above with parameter $\tau_{new}$
and:
\begin{equation}
  w = (\frac{\tau_{old}}{\tau_{new}})^2 \exp[(t_1+t_2)\times(\frac{1}{\tau_{old}}-\frac{1}{\tau_{new}})]
\end{equation}

You will implement this equation in the function `weight_calc` below

In [ ]:
'''
arguements: 
llp_ctau is the list of sum of proper decay length, which is t1+t2 in the equation
new_ctau is tau_new in the equation
old_ctau is tau_old in the equation
All units in cm

return event weight 

'''


def weight_calc(llp_ct, new_ctau, old_ctau):
    denominator = np.exp(-1.0*llp_ct/old_ctau)/old_ctau**2
    numerator = np.exp(-1.0*llp_ct/new_ctau)/new_ctau**2
    weight = numerator/denominator
    
    return weight

**Solution for Ex2 (weight function).** `denominator` is the old per-event probability, so `numerator` is the same expression evaluated with `new_ctau`. Their ratio is `w = [exp(-t/tau_new)/tau_new^2] / [exp(-t/tau_old)/tau_old^2]`, which expands to the `(tau_old/tau_new)^2 exp[t (1/tau_old - 1/tau_new)]` form in the equation above.

Physics intuition: `llp_ct` is `t1 + t2`, the summed proper decay lengths of the two LLPs. The two LLPs decay independently, so the joint probability is a product of two exponentials. It depends only on the sum, and the weight is one number per event.

Common pitfalls: (1) `llp_ct`, `new_ctau` and `old_ctau` must all be in the same units; (2) the weight is only well behaved when the new lifetime is not much longer than the old one, since otherwise a few events dominate the weighted sample; (3) `weight_calc` in `lib/helper.py` implements the same formula and can be used as a cross-check.

## reweight from 10m to 1m

In this exercise you will 
 - use the reweighting function that you have just implemented to reweight the 10m sample to 1m 
 - plot it against the original 1m sample to validate the reweighting function.

You will need to fill in the arguments for the `weight_calc()` function with `llp_ct` and the two lifetimes. 

Be sure to convert everything to cm.

You will then perform an exponential fit to verify that the extracted mean proper decay length of the LLP is 1m for both by replacing the variable `REPLACE_ME`

In [ ]:
%%time
c = rt.TCanvas('c','c', 800, 600)
h = {}
Nevents_todraw = 10000

for i, k in enumerate(['MC_40_1000', 'MC_40_10000']):
    h[k] = rt.TH1D('', '', 100, 0, 1000)
    h[k].SetXTitle('LLP c#tau [cm]')
    h[k].SetYTitle('fraction of events')
    
    if k == 'MC_40_1000': 
        for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0])
    else: 
        llp_ct = np.sum(gLLP_ctau[k], axis = 1)
        
        # Fill in the arguments in weight_calc()
        weight = weight_calc(llp_ct, 100.0, 1000.0) # new = 1 m = 100 cm, old = 10 m = 1000 cm
                
        for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0], weight[j])
    
    h[k].Scale(1./h[k].Integral())
    h[k].SetLineColor(i+1)
    h[k].Draw('hist same')
    

    ####### perform exponential fit ######
    h[k].Fit('expo')
    fit = h[k].GetFunction('expo')
    fitted_slope = fit.GetParameter(1) # expo is exp(p0 + p1*x), so p1 = -1/ctau, in cm
    print("sample: {}, fitted ctau [mm]: {}".format(k, -10.0/fitted_slope))

    #######################################
    
c.SetLogy()
c.Draw()

**Solution for the 10 m to 1 m reweighting.** The 10 m sample has `tau_old = 1000` cm and the target is `tau_new = 100` cm. The call is `weight_calc(llp_ct, 100.0, 1000.0)`, with the arguments in the order `(llp_ct, new_ctau, old_ctau)`. Each event gets its weight. The histogram is filled with the first LLP's `gLLP_ctau`, which gives the marginal distribution of one LLP. The fitted lifetime should come out close to 1000 mm, the same as the native 1 m sample.

Common pitfalls: (1) swapping the new and old arguments gives the inverse weight, so the fit would point to 10 m; (2) use 100.0 and 1000.0 in cm, not 1 and 10 in m or 100 and 1000 in mm; (3) the weight belongs to the event, so fill with `weight[j]` for event `j`, not with a per-LLP weight.

# Ex3 - Calculate the probability of LLP decaying in the muon system

In this exercise you will calculate the probability of the LLP decaying in the muon system by requiring the LLP decay vertex lies within the Muon system.

You will first define the geometric decay region of the muon system based on Fig. 4.1.1 on page 141 of the [Muon Detector Technical Design Report](https://cds.cern.ch/record/343814?ln=en) 

Replace the `REPLACE_ME` in the cell below. 

Then in the last line you will print the probability of LLP decaying in the endcap muon detectors for LLP mean proper decay lenegths (0.1, 1, 10, 100m) that have been generated.

The probability should be on the order of 15-20% for the 1m and 10m sample.

In [ ]:
%%time

for k, T in tree.items():
    
     # define endcap muon system region
    sel_ms = np.abs(T['gLLP_eta'].array()) < 2.4
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) < 1075)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) > 400)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_r'].array()) < 700)
    
    print(k, np.count_nonzero(np.sum(sel_ms, axis = 1))/len(sel_ms))

**Solution for Ex3 (geometric acceptance).** An LLP counts as decaying in the endcap muon system if its decay vertex lies inside a region in `|eta|`, `|z|` and `r`. The selection used here is `|eta| < 2.4` for the muon detector coverage, `400 < |z| < 1075` cm for the endcap station range (1075 cm is where the detector drawings end), and `r < 700` cm for the outer radius of the muon system. Each `np.logical_and` adds one cut, and `np.sum(sel_ms, axis = 1)` counts how many of the two LLPs in each event pass.

Physics intuition: the chance that an LLP decays in a given region depends on its lab-frame decay length, which scales with `beta * gamma * c * tau`. A longer lifetime pushes more decays out to the far endcap, so the acceptance rises with `c * tau`.

Common pitfalls: (1) the boundaries are a choice based on the detector geometry, so check them against the figure the exercise cites; (2) use `np.abs` on z so that both endcaps are included; (3) a rough estimate, with a 1 m sample decay length of about 100 cm, suggests only a few percent of LLPs decay beyond 400 cm, so the 15-20% in the text may refer to the 10 m sample. Compare the printed numbers with that expectation.

## calulate acceptance vs LLP mean proper decay length

Here we will extend on what we just did to calculate the acceptance for intermediate $c\tau$ by calling the 
reweighting function that you've written in the previous exercise to calculate the probability for other intermediate proper decay lengths.


You will fill in the definition of `sel_ms`, which is what you defined in the cell above and the arguments for `weight_calc()` to reweight to intermediate lifetimes in the `ctau_list`

The list of acceptances for different `ctau` is then saved in the `acceptance` list

In [ ]:
%%time

ctau_list  = [100, 300, 500, 1000, 2000, 5000, 10000, 50000, 100000, 500000] #mm

acceptance = []
for ct in ctau_list:
    # find which sample to use for reweighting
    ct_source = 0
    for i in OLD_CTAU:
        if ct <= i: 
            ct_source = i
            break
    if ct_source == 0: ct_source = OLD_CTAU[-1]
    print(ct, ct_source) #verify the correct sample is chosen
    
    T = tree['MC_40_{}'.format(ct_source)]
    
    ################## FILL IN HERE #################
    sel_ms = np.abs(T['gLLP_eta'].array()) < 2.4
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) < 1075)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) > 400)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_r'].array()) < 700)
    llp_ct = np.sum(gLLP_ctau['MC_40_{}'.format(ct_source)], axis = 1)
    weight = weight_calc(llp_ct, ct/10.0, ct_source/10.0) # mm -> cm
    
    #################################################
    
    denominator = np.sum(weight)
    numerator = np.sum(weight[np.sum(sel_ms, axis = 1)>0])
    acceptance.append(numerator/denominator)
    
print(ctau_list, acceptance)

**Solution for the acceptance scan.** `sel_ms` uses the same four cuts as the cell above, applied to the sample named by `ct_source`. The weight is `weight_calc(llp_ct, ct/10.0, ct_source/10.0)`, where `llp_ct` is the summed `gLLP_ctau` of the source sample and both lifetimes are converted from mm to cm. The acceptance is the weighted fraction of events with at least one LLP passing the cuts, `numerator/denominator`, where the denominator is the sum of all weights.

The source sample is the shortest generated lifetime that is at least as long as the target. Reweighting to a shorter lifetime than the source keeps the weights bounded. Reweighting to a much longer lifetime amplifies the weights of the rare long-lived events, which is why a longer source is chosen. The last point (500 m) has no longer sample, so it uses the 100 m sample and has larger weight fluctuations.

Common pitfalls: (1) `weight_calc` needs the lifetimes in cm, so divide the mm values by 10; (2) `T`, `llp_ct` and `sel_ms` must all come from the same `ct_source` sample; (3) normalise by the sum of weights, not by the number of events, otherwise the curve is not normalised to the new lifetime.

## plot acceptance vs LLP mean proper decay length

Here you will plot a TGraph with `ctau_list` on the x-axis and acceptance on the y-axis.

Documentation for TGraph: https://root.cern.ch/doc/master/classTGraph.html



### Q: Do you understand the shape? 

Play around with the `ctau_list` to add more points to make the plot smoother!

In [ ]:
%%time

c = rt.TCanvas('c','c', 800, 800)
c.SetRightMargin(0.04)

h = rt.TGraph(len(ctau_list),array("d", np.array(ctau_list)/1000.0), array("d", acceptance))

h.GetXaxis().SetTitle('c#tau [m]')
h.GetYaxis().SetTitle('acceptance')
h.GetXaxis().SetLimits(0.1,1000.0)
h.GetYaxis().SetRangeUser(1e-3,2)

h.GetXaxis().SetTitleOffset(1)
h.GetYaxis().SetTitleSize(0.05)
h.GetYaxis().SetTitleOffset(1.5)

h.SetLineWidth(3)
h.SetLineColor(4)
h.SetMarkerColor(4)

h.Draw('LPA')

c.SetLogy()
c.SetLogx()
c.SetTicky(1)
c.SetTickx(1)


tdrstyle.setTDRStyle()
c.Draw()